# Фінальний проєкт. Relational Databases for DS

**Совпенко Владислав** · репозиторій `goit-rdb-fp`

Датасет `infectious_cases.csv` — статистика інфекційних захворювань (OWID / WHO) по країнах і регіонах за 1920–2023 рр.: 10 521 рядок, 12 колонок (`Entity`, `Code`, `Year` і 9 показників захворюваності).

План роботи:

1. Схема `pandemic`, імпорт CSV у таблицю `infectious_cases`, огляд даних.
2. Нормалізація до 3НФ: довідник `entities` (Entity + Code) і таблиця показників `cases` з `entity_id`.
3. Агрегати по `Number_rabies` (AVG / MIN / MAX / SUM), топ-10 за середнім.
4. Колонки дат: 1 січня року, поточна дата, різниця в роках через вбудовані функції.
5. Дві власні SQL-функції (`years_since`, `avg_cases_per_period`) і їх використання в запитах.

Ноутбук відтворюваний: у Google Colab PostgreSQL 16 піднімається через `pgserver`, локально можна вказати `FP_PG_URI`.

In [1]:
# ---------- Підключення до PostgreSQL ----------
import os, sys, subprocess, importlib, pathlib

def ensure(pkg, module=None):
    try:
        importlib.import_module(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, mod in [("pandas", None), ("sqlalchemy", None), ("psycopg2-binary", "psycopg2")]:
    ensure(pkg, mod)

import pandas as pd
from sqlalchemy import create_engine, text

PG_URI = os.environ.get("FP_PG_URI")
if PG_URI is None:                       # Google Colab: вбудований PostgreSQL через pgserver
    ensure("pgserver")
    import pgserver
    pg = pgserver.get_server("/content/fp_pg" if os.path.isdir("/content") else "/tmp/fp_pg")
    PG_URI = pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(PG_URI, future=True)

def q(sql):
    """SELECT -> DataFrame (search_path = pandemic у кожному з'єднанні)."""
    with engine.connect() as conn:
        conn.execute(text("SET search_path TO pandemic, public"))
        return pd.read_sql(text(sql), conn)

def run(sql):
    """DDL/DML у транзакції; якщо є результат — повертає DataFrame."""
    with engine.begin() as conn:
        conn.execute(text("SET search_path TO pandemic, public"))
        res = conn.execute(text(sql))
        if res.returns_rows:
            return pd.DataFrame(res.fetchall(), columns=list(res.keys()))

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print(q("SELECT version()").iloc[0, 0])

PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## Завдання 1. Завантаження даних

Створюємо схему `pandemic`, робимо її схемою за замовчуванням через `SET search_path` і завантажуємо CSV так само, як у попередніх домашніх роботах: pandas читає файл і пише його в PostgreSQL через `to_sql`.

Порожні значення в CSV (`,,`) стають `NULL`. Завантажуємо спочатку як текст у `infectious_cases_raw` (точна копія файлу), а потім створюємо типізовану `infectious_cases` з `INTEGER` для року і `NUMERIC` для показників. Це дає змогу в Завданні 5 перевірити, що значення справді числові, ще до приведення типу.

In [2]:
# ---------- 1.1 Схема pandemic + search_path ----------
run("""
DROP SCHEMA IF EXISTS pandemic CASCADE;
CREATE SCHEMA IF NOT EXISTS pandemic;
""")
with engine.begin() as conn:
    conn.execute(text("SET search_path TO pandemic"))
    print("search_path у сесії:", conn.execute(text("SHOW search_path")).scalar())
# далі кожне з'єднання у q()/run() виконує SET search_path TO pandemic, public

search_path у сесії: pandemic


In [3]:
# ---------- 1.2 Отримання CSV ----------
CSV_URL = "https://drive.google.com/uc?export=download&id=1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl"
candidates = [pathlib.Path("data/infectious_cases.csv"), pathlib.Path("../data/infectious_cases.csv"),
              pathlib.Path("infectious_cases.csv")]
csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:                      # у Colab без клону репозиторію — тягнемо з курсового Google Drive
    from urllib.request import urlretrieve
    csv_path = pathlib.Path("infectious_cases.csv")
    urlretrieve(CSV_URL, csv_path)
print("CSV:", csv_path.resolve())

df = pd.read_csv(csv_path, dtype=str, keep_default_na=False).replace("", None)
print(df.shape)
df.head()

CSV: /private/tmp/claude-501/-Users-mac/01519852-a394-483f-96a0-acf33e945251/scratchpad/fp/infectious_cases.csv
(10521, 12)


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,NaN,6160,0,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1981,NaN,5859,0,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1982,NaN,9730,0,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1983,NaN,13937,0,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1984,NaN,3864,0,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ---------- 1.3 Імпорт у PostgreSQL (raw-шар як текст) ----------
df.to_sql("infectious_cases_raw", engine, schema="pandemic", if_exists="replace", index=False)

# ---------- 1.4 Типізована таблиця infectious_cases ----------
run("""
DROP TABLE IF EXISTS infectious_cases;
CREATE TABLE infectious_cases AS
SELECT
    "Entity"                        AS entity,
    "Code"                          AS code,
    "Year"::INTEGER                 AS year,
    "Number_yaws"::NUMERIC          AS number_yaws,
    "polio_cases"::NUMERIC          AS polio_cases,
    "cases_guinea_worm"::NUMERIC    AS cases_guinea_worm,
    "Number_rabies"::NUMERIC        AS number_rabies,
    "Number_malaria"::NUMERIC       AS number_malaria,
    "Number_hiv"::NUMERIC           AS number_hiv,
    "Number_tuberculosis"::NUMERIC  AS number_tuberculosis,
    "Number_smallpox"::NUMERIC      AS number_smallpox,
    "Number_cholera_cases"::NUMERIC AS number_cholera_cases
FROM infectious_cases_raw;
""")
print(q("SELECT COUNT(*) AS rows_raw FROM infectious_cases_raw").to_string(index=False))
print(q("SELECT COUNT(*) AS rows_typed FROM infectious_cases").to_string(index=False))

 rows_raw
    10521
 rows_typed
      10521


In [5]:
# ---------- 1.5 Огляд даних ----------
print(q("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'pandemic' AND table_name = 'infectious_cases'
ORDER BY ordinal_position
""").to_string(index=False))
q("SELECT * FROM infectious_cases ORDER BY entity, year LIMIT 10")

         column_name data_type
              entity      text
                code      text
                year   integer
         number_yaws   numeric
         polio_cases   numeric
   cases_guinea_worm   numeric
       number_rabies   numeric
      number_malaria   numeric
          number_hiv   numeric
 number_tuberculosis   numeric
     number_smallpox   numeric
number_cholera_cases   numeric


,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1960,None,NaN,NaN,None,None,None,None,None,887.0
1,Afghanistan,AFG,1965,None,NaN,NaN,None,None,None,None,None,218.0
2,Afghanistan,AFG,1980,None,6160.0,0.0,None,None,None,None,None,NaN
3,Afghanistan,AFG,1981,None,5859.0,0.0,None,None,None,None,None,NaN
4,Afghanistan,AFG,1982,None,9730.0,0.0,None,None,None,None,None,NaN
5,Afghanistan,AFG,1983,None,13937.0,0.0,None,None,None,None,None,NaN
6,Afghanistan,AFG,1984,None,3864.0,0.0,None,None,None,None,None,NaN
7,Afghanistan,AFG,1985,None,13867.0,0.0,None,None,None,None,None,NaN
8,Afghanistan,AFG,1986,None,12901.0,0.0,None,None,None,None,None,NaN
9,Afghanistan,AFG,1987,None,4396.0,0.0,None,None,None,None,None,NaN


In [6]:
# Повнота колонок, діапазон років, скільки рядків без ISO-коду
q("""
SELECT
    COUNT(*)                                  AS rows_total,
    COUNT(DISTINCT entity)                    AS entities,
    COUNT(*) FILTER (WHERE code IS NULL)      AS rows_without_code,
    MIN(year)                                 AS year_min,
    MAX(year)                                 AS year_max,
    COUNT(number_rabies)                      AS rabies_not_null,
    COUNT(number_malaria)                     AS malaria_not_null,
    COUNT(polio_cases)                        AS polio_not_null,
    COUNT(number_smallpox)                    AS smallpox_not_null
FROM infectious_cases
""")

,rows_total,entities,rows_without_code,year_min,year_max,rabies_not_null,malaria_not_null,polio_not_null,smallpox_not_null
0,10521,245,1128,1920,2023,6780,6780,7610,104


**Спостереження.** У таблиці 10 521 рядок і 245 сутностей; усі рядки завантажились без втрат. Пропусків багато: `Number_rabies` заповнено лише для 6 780 рядків (1990–2019, дані GBD), `Number_smallpox` — для 104. 1 128 рядків мають порожній `Code` — це не країни, а агрегати (`World`, `Africa`, `Lower Middle Income (WB)`, регіони ВООЗ). Пара (`entity`, `year`) унікальна, тож вона є природним ключем рядка.

## Завдання 2. Нормалізація до 3НФ

Ключ вихідної таблиці — (`entity`, `year`). Атрибут `code` залежить лише від `entity` (частина ключа), тобто є часткова залежність, яка порушує 2НФ, а отже й 3НФ. Пара `Entity`/`Code` повторюється в кожному рядку по роках (у `World` — 104 рази).

Рішення: винести довідник `entities (entity_id, entity, code)` і в таблиці показників `cases` зберігати лише `entity_id` + `year` + числові атрибути. Усі неключові атрибути `cases` залежать тільки від повного ключа (`entity_id`, `year`) — 3НФ виконана.

`Code` для агрегатів дорівнює `NULL`, тому для унікальності довідника використано `UNIQUE NULLS NOT DISTINCT` (PostgreSQL 15+): інакше два `NULL` вважалися б різними і дублікати пройшли б.

In [7]:
# ---------- 2.1 Довідник сутностей ----------
run("""
DROP TABLE IF EXISTS cases;
DROP TABLE IF EXISTS entities;

CREATE TABLE entities (
    entity_id  SERIAL PRIMARY KEY,
    entity     TEXT NOT NULL,
    code       TEXT,
    UNIQUE NULLS NOT DISTINCT (entity, code)
);

INSERT INTO entities (entity, code)
SELECT DISTINCT entity, code
FROM infectious_cases
ORDER BY entity, code;
""")
print(q("SELECT COUNT(*) AS entities, COUNT(code) AS with_iso_code, COUNT(*) - COUNT(code) AS aggregates FROM entities").to_string(index=False))
q("SELECT * FROM entities ORDER BY entity_id LIMIT 8")

 entities  with_iso_code  aggregates
      245            211          34


,entity_id,entity,code
0,1,Afghanistan,AFG
1,2,Africa,NaN
2,3,African Region (WHO),NaN
3,4,Albania,ALB
4,5,Algeria,DZA
5,6,American Samoa,ASM
6,7,Americas,NaN
7,8,Andorra,AND


In [8]:
# ---------- 2.2 Таблиця показників з entity_id ----------
run("""
CREATE TABLE cases (
    case_id               SERIAL PRIMARY KEY,
    entity_id             INTEGER NOT NULL REFERENCES entities(entity_id),
    year                  INTEGER NOT NULL CHECK (year BETWEEN 1800 AND 2100),
    number_yaws           NUMERIC,
    polio_cases           NUMERIC,
    cases_guinea_worm     NUMERIC,
    number_rabies         NUMERIC,
    number_malaria        NUMERIC,
    number_hiv            NUMERIC,
    number_tuberculosis   NUMERIC,
    number_smallpox       NUMERIC,
    number_cholera_cases  NUMERIC,
    UNIQUE (entity_id, year)
);

INSERT INTO cases (entity_id, year, number_yaws, polio_cases, cases_guinea_worm, number_rabies,
                   number_malaria, number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases)
SELECT e.entity_id, c.year, c.number_yaws, c.polio_cases, c.cases_guinea_worm, c.number_rabies,
       c.number_malaria, c.number_hiv, c.number_tuberculosis, c.number_smallpox, c.number_cholera_cases
FROM infectious_cases AS c
JOIN entities AS e
  ON e.entity = c.entity
 AND e.code IS NOT DISTINCT FROM c.code;      -- NULL-safe порівняння коду
""")
print(q("SELECT COUNT(*) AS rows_cases FROM cases").to_string(index=False))
q("SELECT * FROM cases ORDER BY case_id LIMIT 5")

 rows_cases
      10521


,case_id,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,1,1,2019,None,32.0,0.0,3.387213,998712.10,849.66254,23717.326,None,2.0
1,2,1,2018,None,23.0,0.0,3.394850,629438.06,810.92230,23930.367,None,NaN
2,3,1,2017,None,16.0,0.0,3.379239,427513.62,765.24567,23483.064,None,33.0
3,4,1,2016,None,14.0,0.0,3.337423,345016.80,718.01410,23112.873,None,677.0
4,5,1,2015,None,22.0,0.0,3.310823,298038.50,669.63860,23044.637,None,58064.0


In [9]:
# Перевірка: нормалізовані таблиці відтворюють оригінал без втрат (JOIN назад = 10 521 рядок, 0 розбіжностей)
q("""
SELECT
    (SELECT COUNT(*) FROM infectious_cases)                                AS original_rows,
    (SELECT COUNT(*) FROM cases c JOIN entities e USING (entity_id))      AS joined_rows,
    (SELECT COUNT(*) FROM (
        SELECT entity, code, year, number_rabies FROM infectious_cases
        EXCEPT
        SELECT e.entity, e.code, c.year, c.number_rabies FROM cases c JOIN entities e USING (entity_id)
    ) d)                                                                   AS rows_lost
""")

,original_rows,joined_rows,rows_lost
0,10521,10521,0


**Спостереження.** Довідник містить 245 рядків (211 країн з ISO-кодом і 34 агрегати без коду), таблиця `cases` — 10 521 рядок, стільки ж, скільки оригінал; `EXCEPT` повертає 0, тобто нормалізація без втрат. `JOIN` з умовою `IS NOT DISTINCT FROM` потрібен саме через `NULL` у `code`: звичайне `=` відкинуло б усі 1 128 рядків агрегатів.

## Завдання 3. Аналіз `Number_rabies`

Для кожної сутності рахуємо середнє, мінімум, максимум і суму показника сказу. Рядки з `NULL` відкидаємо явно, щоб у підсумок не потрапляли сутності без жодного значення. Сортування за середнім у порядку спадання, 10 рядків.

In [10]:
# ---------- 3. AVG / MIN / MAX / SUM по Number_rabies ----------
q("""
SELECT
    e.entity_id,
    e.entity,
    e.code,
    COUNT(c.number_rabies)             AS years_with_data,
    ROUND(AVG(c.number_rabies), 3)     AS avg_rabies,
    ROUND(MIN(c.number_rabies), 3)     AS min_rabies,
    ROUND(MAX(c.number_rabies), 3)     AS max_rabies,
    ROUND(SUM(c.number_rabies), 3)     AS sum_rabies
FROM cases AS c
JOIN entities AS e USING (entity_id)
WHERE c.number_rabies IS NOT NULL
GROUP BY e.entity_id, e.entity, e.code
ORDER BY avg_rabies DESC
LIMIT 10
""")

,entity_id,entity,code,years_with_data,avg_rabies,min_rabies,max_rabies,sum_rabies
0,241,World,OWID_WRL,30,20192.370,14075.508,24744.658,605771.111
1,120,Lower Middle Income (WB),NaN,30,15193.959,10202.528,19182.795,455818.782
2,202,South Asia (WB),NaN,30,11729.889,7271.275,15361.878,351896.663
3,204,South-East Asia Region (WHO),NaN,30,11424.327,6806.013,15641.958,342729.807
4,77,G20,NaN,30,10189.046,6339.080,13164.881,305671.380
5,97,India,IND,30,8599.173,5425.873,11121.142,257975.198
6,209,Sub-Saharan Africa (WB),NaN,30,4748.268,4356.818,5017.977,142448.043
7,3,African Region (WHO),NaN,30,4649.702,4283.073,4917.784,139491.071
8,121,Low Income (WB),NaN,30,3568.105,3014.617,3873.533,107043.164
9,58,East Asia & Pacific (WB),NaN,30,3427.461,2009.940,4591.011,102823.834


**Спостереження.** Топ очолюють агрегати: `World` (середнє 20 192 на рік за 30 років, сума 605 771), далі групи Світового банку й регіони ВООЗ. Єдина країна в десятці — Індія (середнє 8 599, максимум 11 121), на яку припадає близько 43 % світового показника. У кожної сутності рівно 30 років спостережень (1990–2019), тому середні порівнянні між собою. Для аналізу саме країн агрегати можна відсікти умовою `e.code IS NOT NULL`.

In [11]:
# Той самий запит лише по країнах (без агрегатів)
q("""
SELECT e.entity, e.code,
       ROUND(AVG(c.number_rabies), 3) AS avg_rabies,
       ROUND(MIN(c.number_rabies), 3) AS min_rabies,
       ROUND(MAX(c.number_rabies), 3) AS max_rabies,
       ROUND(SUM(c.number_rabies), 3) AS sum_rabies
FROM cases c JOIN entities e USING (entity_id)
WHERE c.number_rabies IS NOT NULL AND e.code IS NOT NULL
GROUP BY e.entity, e.code
ORDER BY avg_rabies DESC
LIMIT 10
""")

,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,World,OWID_WRL,20192.370,14075.508,24744.658,605771.111
1,India,IND,8599.173,5425.873,11121.142,257975.198
2,Pakistan,PAK,1582.170,1177.145,1881.726,47465.089
3,Nigeria,NGA,1335.415,1073.946,1441.119,40062.465
4,China,CHN,1247.907,744.251,2075.593,37437.202
5,Ethiopia,ETH,1145.173,758.050,1323.403,34355.176
6,Myanmar,MMR,972.904,560.697,1449.828,29187.111
7,Bangladesh,BGD,785.709,180.403,1734.708,23571.265
8,Nepal,NPL,661.251,338.135,1041.631,19837.526
9,Philippines,PHL,618.691,358.060,898.071,18560.730


## Завдання 4. Колонка різниці в роках

Використано вбудовані функції PostgreSQL:

- `MAKE_DATE(year, 1, 1)` — дата 1 січня відповідного року;
- `CURRENT_DATE` — поточна дата;
- `AGE(CURRENT_DATE, date)` повертає інтервал, з якого `EXTRACT(YEAR FROM ...)` бере повну кількість років.

`AGE` враховує місяць і день, тож до 1 січня наступного року різниця не збільшується — це «повні роки», а не просто різниця номерів років.

In [12]:
# ---------- 4. Дата 1 січня, поточна дата, різниця в роках ----------
q("""
SELECT
    e.entity,
    c.year,
    MAKE_DATE(c.year, 1, 1)                                             AS year_start_date,
    CURRENT_DATE                                                        AS today,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(c.year, 1, 1)))::INT  AS years_diff
FROM cases AS c
JOIN entities AS e USING (entity_id)
WHERE e.entity = 'Ukraine'
ORDER BY c.year DESC
LIMIT 10
""")

,entity,year,year_start_date,today,years_diff
0,Ukraine,2022,2022-01-01,2026-11-09,4
1,Ukraine,2021,2021-01-01,2026-11-09,5
2,Ukraine,2020,2020-01-01,2026-11-09,6
3,Ukraine,2019,2019-01-01,2026-11-09,7
4,Ukraine,2018,2018-01-01,2026-11-09,8
5,Ukraine,2017,2017-01-01,2026-11-09,9
6,Ukraine,2016,2016-01-01,2026-11-09,10
7,Ukraine,2015,2015-01-01,2026-11-09,11
8,Ukraine,2014,2014-01-01,2026-11-09,12
9,Ukraine,2013,2013-01-01,2026-11-09,13


In [13]:
# Перевірка на крайніх роках датасету
q("""
SELECT DISTINCT year,
       MAKE_DATE(year, 1, 1)                                            AS year_start_date,
       CURRENT_DATE                                                     AS today,
       EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INT AS years_diff
FROM cases
WHERE year IN (SELECT MIN(year) FROM cases) OR year IN (SELECT MAX(year) FROM cases)
ORDER BY year
""")

,year,year_start_date,today,years_diff
0,1920,1920-01-01,2026-11-09,106
1,2023,2023-01-01,2026-11-09,3


## Завдання 5. Власні функції

**`years_since(p_year INTEGER) RETURNS INTEGER`** — та сама логіка, що в Завданні 4, інкапсульована в функцію: рік → дата 1 січня → повних років до поточної дати. Функція `STABLE`, бо залежить від `CURRENT_DATE`, але не змінює даних.

**`avg_cases_per_period(p_cases NUMERIC, p_divisor NUMERIC) RETURNS NUMERIC`** — середня кількість випадків за період: річна кількість, поділена на довільний дільник (12 — місяць, 4 — квартал, 2 — півріччя). Функція `IMMUTABLE`; при нульовому дільнику повертає `NULL` замість помилки ділення на нуль.

Обидві функції написані мовою `SQL`, тому планувальник може інлайнити їх у запит.

In [14]:
# ---------- 5.1 Функція різниці в роках ----------
run("""
CREATE OR REPLACE FUNCTION pandemic.years_since(p_year INTEGER)
RETURNS INTEGER
LANGUAGE sql
STABLE
AS $$
    SELECT EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INTEGER;
$$;
""")
print(q("SELECT years_since(1996) AS years_since_1996, years_since(2023) AS years_since_2023").to_string(index=False))

q("""
SELECT e.entity, c.year,
       MAKE_DATE(c.year, 1, 1)  AS year_start_date,
       years_since(c.year)      AS years_diff_fn,
       EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(c.year, 1, 1)))::INT AS years_diff_builtin
FROM cases c JOIN entities e USING (entity_id)
WHERE e.code = 'UKR'
ORDER BY c.year DESC
LIMIT 5
""")

 years_since_1996  years_since_2023
               30                 3


,entity,year,year_start_date,years_diff_fn,years_diff_builtin
0,Ukraine,2022,2022-01-01,4,4
1,Ukraine,2021,2021-01-01,5,5
2,Ukraine,2020,2020-01-01,6,6
3,Ukraine,2019,2019-01-01,7,7
4,Ukraine,2018,2018-01-01,8,8


In [15]:
# ---------- 5.2 Функція середньої кількості випадків за період ----------
run("""
CREATE OR REPLACE FUNCTION pandemic.avg_cases_per_period(p_cases NUMERIC, p_divisor NUMERIC)
RETURNS NUMERIC
LANGUAGE sql
IMMUTABLE
AS $$
    SELECT CASE
             WHEN p_divisor IS NULL OR p_divisor = 0 THEN NULL
             ELSE ROUND(p_cases / p_divisor, 3)
           END;
$$;
""")
print(q("""
SELECT avg_cases_per_period(1200, 12) AS per_month,
       avg_cases_per_period(1200, 4)  AS per_quarter,
       avg_cases_per_period(1200, 2)  AS per_half_year,
       avg_cases_per_period(1200, 0)  AS zero_divisor
""").to_string(index=False))

 per_month  per_quarter  per_half_year zero_divisor
     100.0        300.0          600.0         None


Умова «лише записи, у яких значення кількості захворювань є числовим» перевіряється двома способами: у типізованій таблиці `cases` колонка вже `NUMERIC`, тож достатньо `IS NOT NULL`; для raw-шару з текстом використано регулярний вираз, який пропускає тільки десяткові числа. Обидва способи дають однакову кількість рядків — 6 780.

In [16]:
# ---------- 5.3 Використання функції в запиті: середнє за місяць / квартал / півріччя ----------
print(q("""
SELECT
    (SELECT COUNT(*) FROM cases WHERE number_rabies IS NOT NULL)                                   AS numeric_typed,
    (SELECT COUNT(*) FROM infectious_cases_raw WHERE "Number_rabies" ~ '^[0-9]+(\\.[0-9]+)?$')     AS numeric_regex_raw
""").to_string(index=False))

q("""
SELECT
    e.entity,
    c.year,
    c.number_rabies                                AS rabies_per_year,
    avg_cases_per_period(c.number_rabies, 12)      AS rabies_per_month,
    avg_cases_per_period(c.number_rabies, 4)       AS rabies_per_quarter,
    avg_cases_per_period(c.number_rabies, 2)       AS rabies_per_half_year,
    years_since(c.year)                            AS years_ago
FROM cases AS c
JOIN entities AS e USING (entity_id)
WHERE c.number_rabies IS NOT NULL                  -- лише числові значення
  AND e.code IN ('IND', 'UKR', 'POL')
  AND c.year >= 2017
ORDER BY e.entity, c.year DESC
""")

 numeric_typed  numeric_regex_raw
          6780               6780


,entity,year,rabies_per_year,rabies_per_month,rabies_per_quarter,rabies_per_half_year,years_ago
0,India,2019,5590.523000,465.877,1397.631,2795.262,7
1,India,2018,5489.927000,457.494,1372.482,2744.964,8
2,India,2017,5425.872600,452.156,1356.468,2712.936,9
3,Poland,2019,0.069733,0.006,0.017,0.035,7
4,Poland,2018,0.067228,0.006,0.017,0.034,8
5,Poland,2017,0.064708,0.005,0.016,0.032,9
6,Ukraine,2019,5.219926,0.435,1.305,2.610,7
7,Ukraine,2018,5.504843,0.459,1.376,2.752,8
8,Ukraine,2017,5.846199,0.487,1.462,2.923,9


In [17]:
# Функція працює і над агрегатами: середньомісячна кількість випадків сказу у світі по роках
q("""
SELECT c.year,
       c.number_rabies                               AS rabies_per_year,
       avg_cases_per_period(c.number_rabies, 12)     AS rabies_per_month,
       avg_cases_per_period(c.number_rabies, 4)      AS rabies_per_quarter,
       avg_cases_per_period(c.number_rabies, 2)      AS rabies_per_half_year
FROM cases c JOIN entities e USING (entity_id)
WHERE e.code = 'OWID_WRL' AND c.number_rabies IS NOT NULL
ORDER BY c.year DESC
LIMIT 6
""")

,year,rabies_per_year,rabies_per_month,rabies_per_quarter,rabies_per_half_year
0,2019,14075.508,1172.959,3518.877,7037.754
1,2018,14204.072,1183.673,3551.018,7102.036
2,2017,14313.008,1192.751,3578.252,7156.504
3,2016,14498.030,1208.169,3624.508,7249.015
4,2015,14742.936,1228.578,3685.734,7371.468
5,2014,15056.748,1254.729,3764.187,7528.374


**Спостереження.** `years_since` збігається з вбудованим виразом із Завдання 4 у кожному рядку. `avg_cases_per_period` для світу у 2019 році дає ≈ 1 173 випадки на місяць, 3 519 на квартал і 7 038 на півріччя; для України значення менші за 1 на місяць — у датасеті це оцінки GBD (дробові), а не зареєстровані випадки. Захист від нульового дільника повертає `NULL`, тому запит не падає з `division by zero`.

In [18]:
# ---------- Підсумок створених об'єктів схеми pandemic ----------
print(q("""
SELECT table_name, table_type
FROM information_schema.tables
WHERE table_schema = 'pandemic'
ORDER BY table_name
""").to_string(index=False))
print()
print(q("""
SELECT routine_name, data_type AS returns
FROM information_schema.routines
WHERE routine_schema = 'pandemic'
ORDER BY routine_name
""").to_string(index=False))

          table_name table_type
               cases BASE TABLE
            entities BASE TABLE
    infectious_cases BASE TABLE
infectious_cases_raw BASE TABLE

        routine_name returns
avg_cases_per_period numeric
         years_since integer


## Висновки

1. Дані імпортовано у схему `pandemic` без втрат: 10 521 рядок, типізовані колонки, порожні значення CSV стали `NULL`.
2. Таблицю нормалізовано до 3НФ: довідник `entities` (245 сутностей, ключ `entity_id`, унікальність пари Entity/Code з урахуванням `NULL`) і таблиця `cases` з зовнішнім ключем та унікальним (`entity_id`, `year`). Зворотний `JOIN` відтворює оригінал повністю.
3. Агрегати по `Number_rabies` показують, що верх рейтингу займають регіональні агрегати, а серед країн лідирує Індія.
4. Різницю в роках побудовано через `MAKE_DATE`, `CURRENT_DATE`, `AGE` і `EXTRACT`.
5. Створено дві функції мовою SQL: `years_since` і `avg_cases_per_period` з захистом від ділення на нуль; обидві використано в запитах лише над числовими значеннями.